# 🚀 The Martian Data Lab
## Activity 1 — Worlds of the Solar System and Beyond

In *The Martian*, Mark Watney is stranded on **Mars** — a cold, thin-atmosphere world where  
liquid water cannot exist on the surface.  But what makes a planet *liveable*?  
Data scientists answer questions like that by collecting measurements, building tables,  
and looking for patterns.

In this notebook you will:
- Build a real data **Table** of planets in our solar system and beyond
- Compute **univariate statistics** (mean, median, min, max)
- Create **visualizations** to spot patterns
- Use data to judge which worlds *might* support life

> 🔬 **Scientist Tip:** Data from NASA's Exoplanet Archive and JPL Solar System Dynamics


In [ ]:
# ── Imports ──────────────────────────────────────────────────
from datascience import *          # Data 8 / Temple course library
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.cm as cm
%matplotlib inline
plt.rcParams.update({'figure.dpi': 110, 'font.size': 11})
print("Libraries loaded! Ready for planetary exploration 🪐")


## Part 1 — Build the Planets Table

The table below contains **12 worlds**: the 8 planets of our solar system  
plus 4 promising **exoplanets** (planets orbiting other stars).

| Column | Meaning |
|--------|---------|
| `Distance (AU)` | Distance from host star — 1 AU = Earth–Sun distance (≈ 150 million km) |
| `Mass (Earth=1)` | Planet mass relative to Earth |
| `Radius (Earth=1)` | Planet radius relative to Earth |
| `Avg Temp (°C)` | Average surface temperature |
| `Has Water?` | Whether liquid water is known or suspected |


In [ ]:
# ── Planet Data ──────────────────────────────────────────────
names = [
    'Mercury', 'Venus', 'Earth', 'Mars',
    'Jupiter', 'Saturn', 'Uranus', 'Neptune',
    'Kepler-452b', 'TRAPPIST-1e', 'Proxima Cen b', 'Kepler-186f'
]

planet_type = ['Solar System'] * 8 + ['Exoplanet'] * 4

# Effective distance from host star in AU
# (adjusted so we can compare across different stellar types)
distance_au = [
    0.387, 0.723, 1.000, 1.524,   # inner solar system
    5.203, 9.537, 19.19, 30.07,   # outer solar system
    1.050, 0.380, 0.320, 0.360    # exoplanets (effective AU)
]

mass_earth = [
    0.055, 0.815,  1.000,  0.107,
    317.8,  95.2,  14.50,  17.10,
      5.0,  0.77,   1.27,   1.44
]

radius_earth = [
    0.383, 0.949,  1.000,  0.532,
    11.21,  9.45,   4.01,   3.88,
     1.60,  0.92,   1.07,   1.17
]

avg_temp_c = [
    167,  464,   15,  -60,
   -110, -140, -195, -200,
     -8,  -22,  -39,  -85
]

has_water = [
    'No', 'No', 'Yes', 'Possibly',
    'No', 'No', 'No', 'No',
    'Possibly', 'Possibly', 'Unknown', 'Possibly'
]

atmosphere = [
    'Minimal', 'CO₂ thick', 'N₂ / O₂', 'CO₂ thin',
    'H₂ / He',  'H₂ / He', 'Ice giant', 'Ice giant',
    'Unknown', 'Unknown', 'Unknown', 'Unknown'
]

rocky = [True, True, True, True,
         False, False, False, False,
         True, True, True, True]

# Build the Table
planets = Table().with_columns(
    'Name',           names,
    'Type',           planet_type,
    'Distance (AU)',  distance_au,
    'Mass (Earth=1)', mass_earth,
    'Radius (Earth=1)', radius_earth,
    'Avg Temp (°C)',  avg_temp_c,
    'Has Water?',     has_water,
    'Atmosphere',     atmosphere,
    'Rocky?',         rocky
)

print(f"Table has {planets.num_rows} rows and {planets.num_columns} columns.")
planets.show()


## Part 2 — Explore the Data

Let's look at the table in different ways before computing any statistics.


In [ ]:
# ── Basic Exploration ────────────────────────────────────────

# Show only the solar system planets
print("=== Solar System Planets ===")
planets.where('Type', 'Solar System').show()

print("\n=== Exoplanets ===")
planets.where('Type', 'Exoplanet').show()


In [ ]:
# ── Find Earth and Mars ──────────────────────────────────────
print("=== Earth ===")
planets.where('Name', 'Earth').show()

print("\n=== Mars (Mark Watney's temporary home) ===")
planets.where('Name', 'Mars').show()

# 🔍 Question: How does Mars compare to Earth in temperature?
#    Write your answer in the cell below as a comment.


## Part 3 — Univariate Statistics

**Univariate** means "one variable at a time." We'll compute summary statistics  
for temperature, mass, and distance.


In [ ]:
# ── Summary Statistics ───────────────────────────────────────
metrics = ['Avg Temp (°C)', 'Mass (Earth=1)', 'Radius (Earth=1)', 'Distance (AU)']

print(f"{'Metric':<22} {'Mean':>9} {'Median':>9} {'Min':>9} {'Max':>9} {'Std Dev':>9}")
print("─" * 70)
for col in metrics:
    vals = planets.column(col)
    print(f"{col:<22} {np.mean(vals):>9.2f} {np.median(vals):>9.2f} "
          f"{np.min(vals):>9.2f} {np.max(vals):>9.2f} {np.std(vals):>9.2f}")


In [ ]:
# ── Rocky planets only ───────────────────────────────────────
rocky_planets = planets.where('Rocky?', True)

print("Statistics for ROCKY planets only:")
print(f"{'Metric':<22} {'Mean':>9} {'Median':>9} {'Min':>9} {'Max':>9}")
print("─" * 58)
for col in ['Avg Temp (°C)', 'Mass (Earth=1)', 'Distance (AU)']:
    vals = rocky_planets.column(col)
    print(f"{col:<22} {np.mean(vals):>9.2f} {np.median(vals):>9.2f} "
          f"{np.min(vals):>9.2f} {np.max(vals):>9.2f}")

print(f"\nNumber of rocky planets/exoplanets: {rocky_planets.num_rows}")


## Part 4 — Visualizations

A good data scientist doesn't just read numbers — they *look* at data.  
Let's make four different plots.


In [ ]:
# ── Plot 1: Temperature Bar Chart ────────────────────────────
fig, ax = plt.subplots(figsize=(12, 5))

names_list = list(planets.column('Name'))
temps      = list(planets.column('Avg Temp (°C)'))
types_list = list(planets.column('Type'))

colors = ['steelblue' if t == 'Solar System' else 'tomato' for t in types_list]
bars   = ax.bar(names_list, temps, color=colors, edgecolor='white', linewidth=0.8)

# Draw 0°C line (freezing)
ax.axhline(0,   color='cyan',  linestyle='--', linewidth=1.5, label='Freezing (0 °C)')
# Shaded "habitable" temperature band
ax.axhspan(-20, 60, alpha=0.12, color='green', label='Comfortable for life (−20 to 60 °C)')

ax.set_ylabel('Average Surface Temperature (°C)', fontsize=12)
ax.set_title('Surface Temperatures of 12 Worlds', fontsize=14, fontweight='bold')
ax.set_xticks(range(len(names_list)))
ax.set_xticklabels(names_list, rotation=40, ha='right')

legend_patches = [
    mpatches.Patch(color='steelblue', label='Solar System'),
    mpatches.Patch(color='tomato',    label='Exoplanet'),
]
ax.legend(handles=legend_patches + [
    plt.Line2D([0],[0], color='cyan',  linestyle='--', label='Freezing (0 °C)'),
    mpatches.Patch(color='green', alpha=0.3, label='Comfortable for life'),
], loc='upper right', fontsize=9)

plt.tight_layout()
plt.savefig('/home/claude/plot_temperatures.png', dpi=120)
plt.show()
print("\n🔍 Which planets fall inside the green 'comfortable for life' band?")


In [ ]:
# ── Plot 2: Distance vs Temperature (Scatter) ────────────────
fig, ax = plt.subplots(figsize=(10, 6))

color_map = {'Solar System': 'steelblue', 'Exoplanet': 'tomato'}
marker_map = {True: 'o', False: 's'}   # circle=rocky, square=gas giant

for row in planets.rows:
    name   = row[0]
    ptype  = row[1]
    dist   = row[2]
    temp   = row[5]
    is_rocky = row[8]

    sc = ax.scatter(dist, temp,
                    c=color_map[ptype],
                    marker=marker_map[is_rocky],
                    s=120, edgecolors='white', linewidth=0.8, zorder=3)
    ax.annotate(name, (dist, temp),
                textcoords='offset points', xytext=(6, 4),
                fontsize=8, color='white',
                bbox=dict(boxstyle='round,pad=0.2', fc='#1a1a2e', alpha=0.6))

# Habitable temperature band
ax.axhspan(-20, 60, alpha=0.15, color='green', label='Habitable temp range')
ax.axhline(0,  color='cyan', linestyle='--', linewidth=1, alpha=0.7)

ax.set_facecolor('#1a1a2e')
fig.patch.set_facecolor('#0f0f23')
ax.tick_params(colors='white')
ax.xaxis.label.set_color('white')
ax.yaxis.label.set_color('white')
ax.title.set_color('white')
for spine in ax.spines.values():
    spine.set_edgecolor('#444466')

ax.set_xlabel('Distance from Star (AU)', fontsize=12)
ax.set_ylabel('Average Surface Temperature (°C)', fontsize=12)
ax.set_title('Distance from Star vs Surface Temperature', fontsize=13, fontweight='bold')
ax.set_xscale('log')

legend_elems = [
    mpatches.Patch(color='steelblue', label='Solar System'),
    mpatches.Patch(color='tomato',    label='Exoplanet'),
    plt.scatter([], [], marker='o', c='white', s=80, label='Rocky'),
    plt.scatter([], [], marker='s', c='white', s=80, label='Gas/Ice Giant'),
    mpatches.Patch(color='green', alpha=0.3, label='Habitable temp range'),
]
ax.legend(handles=legend_elems, fontsize=9, facecolor='#1a1a2e',
          labelcolor='white', edgecolor='#444466')

plt.tight_layout()
plt.savefig('/home/claude/plot_scatter.png', dpi=120, facecolor=fig.get_facecolor())
plt.show()
print("\n🔍 Describe the general trend: as distance increases, what happens to temperature?")


In [ ]:
# ── Plot 3: Mass vs Radius ───────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 6))

masses  = planets.column('Mass (Earth=1)')
radii   = planets.column('Radius (Earth=1)')
types_col = planets.column('Type')
rocky_col = planets.column('Rocky?')

for i in range(len(names)):
    c = 'steelblue' if types_col[i] == 'Solar System' else 'tomato'
    m = 'o' if rocky_col[i] else 's'
    ax.scatter(masses[i], radii[i], c=c, marker=m, s=130,
               edgecolors='white', linewidth=0.7, zorder=3)
    ax.annotate(names[i], (masses[i], radii[i]),
                textcoords='offset points', xytext=(5, 3), fontsize=8)

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Mass (Earths)', fontsize=12)
ax.set_ylabel('Radius (Earths)', fontsize=12)
ax.set_title('Planet Mass vs Radius', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3)

# Highlight Earth
earth_idx = names.index('Earth')
ax.scatter(masses[earth_idx], radii[earth_idx],
           c='limegreen', marker='*', s=300, zorder=5, label='Earth')
ax.legend(fontsize=9)

plt.tight_layout()
plt.show()
print("\n🔍 What pattern do you notice between mass and radius for the giant planets?")


In [ ]:
# ── Plot 4: The Goldilocks Zone ──────────────────────────────
fig, ax = plt.subplots(figsize=(12, 5))

# Distance color-coded by temperature for solar system
ss = planets.where('Type', 'Solar System')
ex = planets.where('Type', 'Exoplanet')

temps_all  = planets.column('Avg Temp (°C)')
norm       = plt.Normalize(vmin=temps_all.min(), vmax=temps_all.max())
cmap       = plt.cm.RdYlBu_r

y_ss = np.zeros(ss.num_rows)
y_ex = np.ones(ex.num_rows)

sc1 = ax.scatter(ss.column('Distance (AU)'), y_ss,
                 c=ss.column('Avg Temp (°C)'), cmap=cmap, norm=norm,
                 s=200, edgecolors='white', linewidth=1, zorder=3)
sc2 = ax.scatter(ex.column('Distance (AU)'), y_ex,
                 c=ex.column('Avg Temp (°C)'), cmap=cmap, norm=norm,
                 s=200, edgecolors='white', linewidth=1, marker='^', zorder=3)

for row in ss.rows:
    ax.annotate(row[0], (row[2], 0), textcoords='offset points',
                xytext=(0, -18), ha='center', fontsize=8)
for row in ex.rows:
    ax.annotate(row[0], (row[2], 1), textcoords='offset points',
                xytext=(0, 10), ha='center', fontsize=8)

# Goldilocks zone (approximate, Sun-like star)
ax.axvspan(0.85, 1.70, alpha=0.18, color='green', label='Goldilocks Zone (Sun-like)')

ax.set_xlim(0.1, 40)
ax.set_xscale('log')
ax.set_yticks([0, 1])
ax.set_yticklabels(['Solar System', 'Exoplanets'], fontsize=11)
ax.set_xlabel('Distance from Star (AU)', fontsize=12)
ax.set_title('The Goldilocks Zone — Not Too Hot, Not Too Cold', fontsize=13, fontweight='bold')

cbar = plt.colorbar(sc1, ax=ax, pad=0.02)
cbar.set_label('Surface Temp (°C)', fontsize=10)
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('/home/claude/plot_goldilocks.png', dpi=120)
plt.show()


## Part 5 — Habitability Score

Let's build a simple **Habitability Index** using three criteria:
1. **Temperature** — must be between −20 °C and 60 °C for liquid water
2. **Rocky** — gas giants can't support surface life
3. **Water** — presence or possibility of liquid water scores extra points


In [ ]:
# ── Compute Habitability Score (0–10) ────────────────────────
def habitability_score(temp_c, is_rocky, water_status):
    score = 0
    
    # Temperature scoring (up to 5 points)
    if -20 <= temp_c <= 60:
        temp_score = 5 - abs(temp_c - 20) / 16   # optimal ~20°C
        score += max(0, temp_score)
    elif -60 <= temp_c < -20 or 60 < temp_c <= 100:
        score += 1   # marginal
    # else: 0
    
    # Rocky bonus (2 points)
    if is_rocky:
        score += 2
    
    # Water bonus (3 points)
    water_bonus = {'Yes': 3, 'Possibly': 2, 'Unknown': 1, 'No': 0}
    score += water_bonus.get(water_status, 0)
    
    return round(min(score, 10), 2)

# Apply to every planet
scores = []
for row in planets.rows:
    temp   = row[5]
    rocky  = row[8]
    water  = row[6]
    scores.append(habitability_score(temp, rocky, water))

planets = planets.with_column('Habitability (0-10)', scores)

# Sort by habitability
sorted_planets = planets.sort('Habitability (0-10)', descending=True)
sorted_planets.select('Name', 'Type', 'Avg Temp (°C)', 'Has Water?',
                       'Rocky?', 'Habitability (0-10)').show()


In [ ]:
# ── Habitability Bar Chart ────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 5))

sorted_names   = list(sorted_planets.column('Name'))
sorted_scores  = list(sorted_planets.column('Habitability (0-10)'))
sorted_types   = list(sorted_planets.column('Type'))

bar_colors = []
for s in sorted_scores:
    if   s >= 7:   bar_colors.append('#2ecc71')   # green
    elif s >= 4:   bar_colors.append('#f39c12')   # orange
    elif s >= 1:   bar_colors.append('#e74c3c')   # red
    else:          bar_colors.append('#7f8c8d')   # grey

bars = ax.bar(sorted_names, sorted_scores, color=bar_colors,
              edgecolor='white', linewidth=0.8)

# Value labels on bars
for bar, score in zip(bars, sorted_scores):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
            f'{score:.1f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

ax.set_ylim(0, 11)
ax.set_ylabel('Habitability Score (0–10)', fontsize=12)
ax.set_title('Planetary Habitability Index', fontsize=14, fontweight='bold')
ax.set_xticks(range(len(sorted_names)))
ax.set_xticklabels(sorted_names, rotation=35, ha='right')

legend_patches = [
    mpatches.Patch(color='#2ecc71', label='Good (≥ 7)'),
    mpatches.Patch(color='#f39c12', label='Marginal (4–7)'),
    mpatches.Patch(color='#e74c3c', label='Poor (1–4)'),
    mpatches.Patch(color='#7f8c8d', label='Uninhabitable'),
]
ax.legend(handles=legend_patches, loc='upper right', fontsize=9)
plt.tight_layout()
plt.show()


## 🧠 Reflection Questions

Answer these questions in the **next cell** as Python comments (`# ...`) or in a new Markdown cell.

1. **Mars vs Earth**: What are the two biggest differences (by data) between Mars and Earth  
   that make Mars difficult for Mark Watney to survive?

2. **Best exoplanet candidate**: Based on your habitability score, which exoplanet  
   would be the *best* candidate to find life? What data supports your answer?

3. **Temperature pattern**: Describe the relationship between a planet's distance  
   from its star and its surface temperature. Is it a straight-line relationship or curved?

4. **Improve the score**: What other properties would you add to make the habitability  
   index better? How would you measure them?

5. **The Martian connection**: In the book, Watney grows potatoes in Mars soil.  
   What temperature and water challenges would he need to overcome, according to our data?


In [ ]:
# Write your answers here as comments:

# Q1: Mars vs Earth differences:
#

# Q2: Best exoplanet candidate:
#

# Q3: Temperature–distance relationship:
#

# Q4: Improvements to habitability score:
#

# Q5: The Martian connection:
#


---
## 🏆 Bonus Challenge

Write code below to:
1. Filter for only planets with `Avg Temp (°C)` between **−50 and 100**
2. Of those, select only rocky planets
3. Sort by **distance from Earth** (hint: Earth is at 1.0 AU — compute the absolute difference)
4. Display the top 3 candidates

*Hint:* You can create a new column with `planets.with_column('name', array)`


In [ ]:
# 🏆 Bonus Challenge — your code here:

